## MitoEye Validation

In [1]:
# %% [markdown]
# # Multi-fold Segmentation Evaluation Pipeline - 2D Blue Channel Version

# %%
import json
import re
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import tifffile
from sklearn.metrics import cohen_kappa_score

# — Ensure pandas will display full DataFrames and string columns —
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

def scan_tiffs(base_dirs: list[Path]) -> pd.DataFrame:
    """
    Scan for .tif/.tiff files across multiple directories and return a DataFrame
    with one row per file: folder, filename, width, height, depth, channels.
    """
    print(f"Scanning TIFFs across {len(base_dirs)} directories...")
    records = []
    
    for base_dir in base_dirs:
        print(f"\nScanning {base_dir.name}...")
        # Only look for non-overlay TIFFs (original files) - CHANGED TO 2D_OVERLAY
        tiff_files = [f for f in base_dir.glob('*.tiff') if '2D_OVERLAY' not in f.name]
        tiff_files.extend([f for f in base_dir.glob('*.tif') if '2D_OVERLAY' not in f.name])
        
        for tif_path in tiff_files:
            print(f"  Reading {tif_path.name}")
            try:
                with tifffile.TiffFile(str(tif_path)) as tif:
                    depth = len(tif.pages)
                    arr0 = tif.pages[0].asarray()
                    H, W = arr0.shape[:2]
                    C = arr0.shape[2] if arr0.ndim == 3 else 1
            except Exception as e:
                print(f"    ❌ Could not read {tif_path.name}: {e}")
                continue
            
            records.append({
                'folder': base_dir.name,
                'filename': tif_path.name,
                'width': W,
                'height': H,
                'depth': depth,
                'channels': C,
            })
    
    df = pd.DataFrame(records)
    print(f"\nScanned {len(df)} TIFF files total.")
    print("\nDisplaying full table:")
    display(df)
    return df


def extract_fold_from_overlay(filename: str) -> int:
    """
    Extract fold number from overlay filename.
    E.g., "B3Gt061_1_A5v2_78-205_combined_2D_OVERLAY_model4005_fold1.tiff" -> 1
    """
    match = re.search(r'fold(\d+)', filename)
    if match:
        return int(match.group(1))
    return None


def compute_metrics_for_splits(
    split_dirs: list[Path],
    out_dir: Path
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    For each split directory, compute SENS, SPEC, ACC, DSC, AVD(%), Kappa(κ)
    by comparing ground truth (BLUE channel of original) with predictions
    (BLUE channel of overlay). Writes out two CSVs under `out_dir`:
      - metrics_per_file_<timestamp>.csv
      - metrics_summary_by_fold_<timestamp>.csv
    Returns (df_all, df_summary).
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    all_records = []
    
    for split_dir in split_dirs:
        split_name = split_dir.name
        fold_num = int(split_name.replace('split', ''))
        
        print(f"\n{'='*60}")
        print(f"Processing {split_name} (fold {fold_num})")
        print(f"{'='*60}")
        
        # Find all overlay files in this split - CHANGED TO 2D_OVERLAY
        overlay_files = list(split_dir.glob('*2D_OVERLAY*.tiff'))
        overlay_files.extend(list(split_dir.glob('*2D_OVERLAY*.tif')))
        
        print(f"Found {len(overlay_files)} overlay files.")
        
        for i, overlay_path in enumerate(overlay_files, 1):
            overlay_name = overlay_path.name
            print(f"\n  [{i}/{len(overlay_files)}] Processing: {overlay_name}")
            
            # Find corresponding original file (without overlay suffix)
            # Remove the overlay suffix pattern to get the base name - CHANGED TO 2D_OVERLAY
            base_name = re.sub(r'_2D_OVERLAY_model\d+_fold\d+', '', overlay_path.stem)
            
            # Look for the original file
            orig_path = None
            for ext in ['.tiff', '.tif']:
                candidate = split_dir / f"{base_name}{ext}"
                if candidate.exists():
                    orig_path = candidate
                    break
            
            if orig_path is None:
                print(f"    ❌ Original file not found for {overlay_name}")
                continue
            
            print(f"    Found original: {orig_path.name}")
            
            try:
                # Load original file
                orig_arr = tifffile.imread(str(orig_path))
                print(f"    Original shape: {orig_arr.shape}")
                
                # Extract ground truth from RED channel (channel 0)
                if orig_arr.ndim == 4 and orig_arr.shape[-1] >= 3:
                    # 3D volume with RGB channels
                    gt = orig_arr[..., 0]  # Red channel for ground truth
                elif orig_arr.ndim == 3 and orig_arr.shape[-1] >= 3:
                    # 2D image with RGB channels
                    gt = orig_arr[..., 0][np.newaxis, ...]  # Red channel, add depth dimension
                elif orig_arr.ndim == 3 and orig_arr.shape[-1] == 1:
                    # 3D volume with single channel
                    gt = orig_arr[..., 0]
                elif orig_arr.ndim == 2:
                    # 2D single channel
                    gt = orig_arr[np.newaxis, ...]
                else:
                    gt = orig_arr
                
                # Load overlay file
                overlay_arr = tifffile.imread(str(overlay_path))
                print(f"    Overlay shape: {overlay_arr.shape}")
                
                # Extract prediction from BLUE channel (channel 2)
                if overlay_arr.ndim == 4 and overlay_arr.shape[-1] >= 3:
                    # 3D volume with RGB channels
                    pred = overlay_arr[..., 2]  # Blue channel for predictions
                elif overlay_arr.ndim == 3 and overlay_arr.shape[-1] >= 3:
                    # 2D image with RGB channels
                    pred = overlay_arr[..., 2][np.newaxis, ...]  # Blue channel, add depth dimension
                elif overlay_arr.ndim == 3 and overlay_arr.shape[-1] == 1:
                    # 3D volume with single channel
                    pred = overlay_arr[..., 0]
                elif overlay_arr.ndim == 2:
                    # 2D single channel
                    pred = overlay_arr[np.newaxis, ...]
                else:
                    pred = overlay_arr
                
                # Ensure same shape
                if gt.shape != pred.shape:
                    print(f"    ⚠ Shape mismatch: GT {gt.shape} vs Pred {pred.shape}")
                    # Try to match shapes
                    min_depth = min(gt.shape[0], pred.shape[0])
                    gt = gt[:min_depth]
                    pred = pred[:min_depth]
                
                # Normalize and threshold
                gt = gt.astype(np.float32)
                if gt.max() > 1.0:
                    gt /= gt.max()
                gt_mask = gt > 0.5
                
                pred = pred.astype(np.float32)
                if pred.max() > 1.0:
                    pred /= pred.max()
                pred_mask = pred > 0.5
                
                # Compute metrics
                g = gt_mask.ravel()
                p = pred_mask.ravel()
                
                TP = np.logical_and(g, p).sum()
                TN = np.logical_and(~g, ~p).sum()
                FP = np.logical_and(~g, p).sum()
                FN = np.logical_and(g, ~p).sum()
                total = g.size
                
                SENS = round(TP/(TP+FN) if TP+FN > 0 else 0.0, 4)
                SPEC = round(TN/(TN+FP) if TN+FP > 0 else 0.0, 4)
                ACC = round((TP+TN)/total if total > 0 else 0.0, 4)
                DSC = round(2*TP/(2*TP+FP+FN) if (2*TP+FP+FN) > 0 else 0.0, 4)
                AVD = round(abs(g.sum()-p.sum())/g.sum()*100 if g.sum() > 0 else 0.0, 4)
                KAPPA = round(cohen_kappa_score(g, p), 4)
                
                all_records.append({
                    'fold': fold_num,
                    'split': split_name,
                    'filename': overlay_name,
                    'original': orig_path.name,
                    'SENS': SENS,
                    'SPEC': SPEC,
                    'ACC': ACC,
                    'DSC': DSC,
                    'AVD (%)': AVD,
                    'Kappa (κ)': KAPPA
                })
                
                print(f"    ✓ Metrics: SENS={SENS:.3f}, SPEC={SPEC:.3f}, ACC={ACC:.3f}, DSC={DSC:.3f}, AVD={AVD:.2f}%, κ={KAPPA:.3f}")
                
            except Exception as e:
                print(f"    ❌ Error processing files: {e}")
                continue
    
    # Create DataFrame
    df_all = pd.DataFrame(all_records)
    
    if len(df_all) > 0:
        # Table 1: per-file results with overall average
        metrics_cols = ['SENS', 'SPEC', 'ACC', 'DSC', 'AVD (%)', 'Kappa (κ)']
        avg = df_all[metrics_cols].mean().round(4)
        avg_row = {
            'fold': 'Overall',
            'split': 'Average',
            'filename': '-',
            'original': '-',
            **avg.to_dict()
        }
        df_all_with_avg = pd.concat([df_all, pd.DataFrame([avg_row])], ignore_index=True)
        
        # Save per-file metrics
        table1_fp = out_dir / f'metrics_per_file_2D_blue_{ts}.csv'  # Added 2D_blue to filename
        df_all_with_avg.to_csv(table1_fp, index=False)
        print(f"\n✅ Saved per-file metrics to {table1_fp}")
        
        # Table 2: per-fold summary
        per_fold = (
            df_all.groupby(['fold', 'split'])[metrics_cols]
            .agg(['mean', 'std', 'count'])
            .round(4)
        )
        
        # Flatten multi-level columns
        per_fold.columns = ['_'.join(col).strip() for col in per_fold.columns.values]
        per_fold = per_fold.reset_index()
        
        # Add overall summary
        overall_stats = {}
        for metric in metrics_cols:
            overall_stats[f'{metric}_mean'] = df_all[metric].mean()
            overall_stats[f'{metric}_std'] = df_all[metric].std()
            overall_stats[f'{metric}_count'] = len(df_all)
        
        overall_row = pd.DataFrame([{
            'fold': 'Overall',
            'split': 'All',
            **overall_stats
        }])
        
        df_summary = pd.concat([per_fold, overall_row], ignore_index=True)
        
        # Save fold summary
        table2_fp = out_dir / f'metrics_summary_by_fold_2D_blue_{ts}.csv'  # Added 2D_blue to filename
        df_summary.to_csv(table2_fp, index=False)
        print(f"✅ Saved fold-summary metrics to {table2_fp}")
        
        # Display results
        print("\n" + "="*80)
        print("TABLE 1: Per-file Metrics (2D Blue Channel)")
        print("="*80)
        display(df_all_with_avg)
        
        print("\n" + "="*80)
        print("TABLE 2: Summary by Fold (2D Blue Channel)")
        print("="*80)
        display(df_summary)
        
        # Create simplified summary table showing just means
        simple_summary = df_all.groupby('fold')[metrics_cols].mean().round(4)
        overall_means = df_all[metrics_cols].mean().round(4)
        overall_means.name = 'Overall'
        simple_summary = pd.concat([simple_summary, overall_means.to_frame().T])
        
        print("\n" + "="*80)
        print("TABLE 3: Simplified Summary (Means Only) - 2D Blue Channel")
        print("="*80)
        display(simple_summary)
        
        # Save simplified summary
        table3_fp = out_dir / f'metrics_simple_summary_2D_blue_{ts}.csv'  # Added 2D_blue to filename
        simple_summary.to_csv(table3_fp)
        print(f"✅ Saved simplified summary to {table3_fp}")
        
    else:
        print("\n⚠ No valid data processed!")
        df_all_with_avg = df_all
        df_summary = pd.DataFrame()
    
    return df_all_with_avg, df_summary

In [2]:
# %%
# Define paths
base_path = Path('/media/home/DATA10TB/MITOCHONDRIA/TRAINING_SET_FINAL')
split_dirs = [base_path / f'split{i}' for i in range(1, 6)]
out_dir = Path('./evaluation_results_2D_blue')  # Changed output directory

# Verify all directories exist
for split_dir in split_dirs:
    if not split_dir.exists():
        print(f"⚠ Warning: {split_dir} does not exist!")
    else:
        print(f"✓ Found: {split_dir}")

# Scan TIFFs across all splits (optional, for overview)
print("\n" + "="*80)
print("SCANNING ALL TIFF FILES (Excluding 2D_OVERLAY files)")
print("="*80)
df_scan = scan_tiffs(split_dirs)

# Compute metrics for all splits
print("\n" + "="*80)
print("COMPUTING METRICS FOR ALL SPLITS (2D Blue Channel)")
print("="*80)
df_all, df_summary = compute_metrics_for_splits(split_dirs, out_dir)

print("\n" + "="*80)
print("EVALUATION COMPLETE! (2D Blue Channel Version)")
print("="*80)
print(f"Results saved to: {out_dir}")

✓ Found: /media/home/DATA10TB/MITOCHONDRIA/TRAINING_SET_FINAL/split1
✓ Found: /media/home/DATA10TB/MITOCHONDRIA/TRAINING_SET_FINAL/split2
✓ Found: /media/home/DATA10TB/MITOCHONDRIA/TRAINING_SET_FINAL/split3
✓ Found: /media/home/DATA10TB/MITOCHONDRIA/TRAINING_SET_FINAL/split4
✓ Found: /media/home/DATA10TB/MITOCHONDRIA/TRAINING_SET_FINAL/split5

SCANNING ALL TIFF FILES (Excluding 2D_OVERLAY files)
Scanning TIFFs across 5 directories...

Scanning split1...
  Reading B3Gt061PbV3_A7_converted_381-508_combined.tiff
  Reading B3Gt061PbV3_A7_converted_381-508_combined_3D_OVERLAY_model4005_fold0.tiff
  Reading B3Gt061_1_A5v2_78-205_combined.tiff
  Reading B3Gt061_1_A5v2_78-205_combined_3D_OVERLAY_model4005_fold0.tiff
  Reading B3GT101_220825-1-128_combined.tiff
  Reading B3GT101_220825-1-128_combined_3D_OVERLAY_model4005_fold0.tiff
  Reading B3GT101_220920_converted_-1-128_combined.tiff
  Reading B3GT101_220920_converted_-1-128_combined_3D_OVERLAY_model4005_fold0.tiff

Scanning split2...
  Read

,folder,filename,width,height,depth,channels
0,split1,B3Gt061PbV3_A7_converted_381-508_combined.tiff,2671,957,128,3
1,split1,B3Gt061PbV3_A7_converted_381-508_combined_3D_OVERLAY_model4005_fold0.tiff,2671,957,128,3
2,split1,B3Gt061_1_A5v2_78-205_combined.tiff,2666,996,128,3
3,split1,B3Gt061_1_A5v2_78-205_combined_3D_OVERLAY_model4005_fold0.tiff,2666,996,128,3
4,split1,B3GT101_220825-1-128_combined.tiff,2665,995,128,3
5,split1,B3GT101_220825-1-128_combined_3D_OVERLAY_model4005_fold0.tiff,2665,995,128,3
6,split1,B3GT101_220920_converted_-1-128_combined.tiff,2675,1008,128,3
7,split1,B3GT101_220920_converted_-1-128_combined_3D_OVERLAY_model4005_fold0.tiff,2675,1008,128,3
8,split2,B3GT084_BB1_170722-a_-1-128_combined.tiff,2667,1346,128,3
9,split2,B3GT084_BB1_170722-a_-1-128_combined_3D_OVERLAY_model4005_fold1.tiff,2667,1346,128,3



COMPUTING METRICS FOR ALL SPLITS (2D Blue Channel)

Processing split1 (fold 1)
Found 4 overlay files.

  [1/4] Processing: B3Gt061PbV3_A7_converted_381-508_combined_2D_OVERLAY_model2005_fold0.tiff
    Found original: B3Gt061PbV3_A7_converted_381-508_combined.tiff
    Original shape: (128, 957, 2671, 3)
    Overlay shape: (128, 957, 2671, 3)
    ✓ Metrics: SENS=0.806, SPEC=0.942, ACC=0.930, DSC=0.663, AVD=43.10%, κ=0.626

  [2/4] Processing: B3Gt061_1_A5v2_78-205_combined_2D_OVERLAY_model2005_fold0.tiff
    Found original: B3Gt061_1_A5v2_78-205_combined.tiff
    Original shape: (128, 996, 2666, 3)
    Overlay shape: (128, 996, 2666, 3)
    ✓ Metrics: SENS=0.882, SPEC=0.986, ACC=0.976, DSC=0.879, AVD=0.66%, κ=0.866

  [3/4] Processing: B3GT101_220825-1-128_combined_2D_OVERLAY_model2005_fold0.tiff
    Found original: B3GT101_220825-1-128_combined.tiff
    Original shape: (128, 995, 2665, 3)
    Overlay shape: (128, 995, 2665, 3)
    ✓ Metrics: SENS=0.971, SPEC=0.986, ACC=0.985, DSC=0.904

,fold,split,filename,original,SENS,SPEC,ACC,DSC,AVD (%),Kappa (κ)
0,1,split1,B3Gt061PbV3_A7_converted_381-508_combined_2D_OVERLAY_model2005_fold0.tiff,B3Gt061PbV3_A7_converted_381-508_combined.tiff,0.8065,0.9419,0.9304,0.6635,43.0978,0.6260
1,1,split1,B3Gt061_1_A5v2_78-205_combined_2D_OVERLAY_model2005_fold0.tiff,B3Gt061_1_A5v2_78-205_combined.tiff,0.8823,0.9864,0.9761,0.8794,0.6614,0.8661
2,1,split1,B3GT101_220825-1-128_combined_2D_OVERLAY_model2005_fold0.tiff,B3GT101_220825-1-128_combined.tiff,0.9706,0.9857,0.9846,0.9036,14.8432,0.8953
3,1,split1,B3GT101_220920_converted_-1-128_combined_2D_OVERLAY_model2005_fold0.tiff,B3GT101_220920_converted_-1-128_combined.tiff,0.9678,0.9842,0.9814,0.9466,4.4894,0.9353
4,2,split2,B3GT084_BB1_170722-a_-1-128_combined_2D_OVERLAY_model2005_fold1.tiff,B3GT084_BB1_170722-a_-1-128_combined.tiff,0.8689,0.9951,0.9831,0.9074,8.4803,0.8981
5,2,split2,B3GT084_Bb2a_170415-a_-13-140_combined_2D_OVERLAY_model2005_fold1.tiff,B3GT084_Bb2a_170415-a_-13-140_combined.tiff,0.8269,0.9975,0.9837,0.8915,14.4863,0.8827
6,2,split2,B3GT109_reda_170818-a_-30-157_combined_2D_OVERLAY_model2005_fold1.tiff,B3GT109_reda_170818-a_-30-157_combined.tiff,0.8485,0.9980,0.9893,0.9020,11.8637,0.8964
7,2,split2,B3GT109_redb_170816-a-1-128_combined_2D_OVERLAY_model2005_fold1.tiff,B3GT109_redb_170816-a-1-128_combined.tiff,0.8044,0.9973,0.9837,0.8741,15.9326,0.8654
8,3,split3,B3Gt029B1_a2_try3_731-858_combined_2D_OVERLAY_model2005_fold2.tiff,B3Gt029B1_a2_try3_731-858_combined.tiff,0.9379,0.9913,0.9850,0.9364,0.3185,0.9279
9,3,split3,B3Gt029B1_a3b_try4_20-147_combined_2D_OVERLAY_model2005_fold2.tiff,B3Gt029B1_a3b_try4_20-147_combined.tiff,0.8794,0.9900,0.9830,0.8664,3.0003,0.8574



TABLE 2: Summary by Fold (2D Blue Channel)


,fold,split,SENS_mean,SENS_std,SENS_count,SPEC_mean,SPEC_std,SPEC_count,ACC_mean,ACC_std,ACC_count,DSC_mean,DSC_std,DSC_count,AVD (%)_mean,AVD (%)_std,AVD (%)_count,Kappa (κ)_mean,Kappa (κ)_std,Kappa (κ)_count
0,1,split1,0.90680,0.078400,4,0.97460,0.021800,4,0.968100,0.025400,4,0.84830,0.126300,4,15.77300,19.17630,4,0.83070,0.13940,4
1,2,split2,0.83720,0.027800,4,0.99700,0.001300,4,0.985000,0.002900,4,0.89380,0.014700,4,12.69070,3.27340,4,0.88560,0.01520,4
2,3,split3,0.91260,0.037700,4,0.99150,0.001700,4,0.984000,0.005700,4,0.91130,0.033200,4,3.02240,2.11530,4,0.90240,0.03470,4
3,4,split4,0.79810,0.104400,4,0.97800,0.012100,4,0.957700,0.017200,4,0.81090,0.107700,4,5.38130,4.38740,4,0.78650,0.11220,4
4,5,split5,0.94660,0.032900,4,0.98660,0.017200,4,0.983200,0.013300,4,0.93110,0.028300,4,7.46340,8.21300,4,0.92140,0.03630,4
5,Overall,All,0.88026,0.079481,20,0.98552,0.014768,20,0.975595,0.017545,20,0.87907,0.081819,20,8.86616,9.87083,20,0.86534,0.08995,20



TABLE 3: Simplified Summary (Means Only) - 2D Blue Channel


,SENS,SPEC,ACC,DSC,AVD (%),Kappa (κ)
1,0.9068,0.9746,0.9681,0.8483,15.7730,0.8307
2,0.8372,0.9970,0.9850,0.8938,12.6907,0.8856
3,0.9126,0.9915,0.9840,0.9113,3.0224,0.9024
4,0.7981,0.9780,0.9577,0.8109,5.3813,0.7865
5,0.9466,0.9866,0.9832,0.9311,7.4634,0.9214
Overall,0.8803,0.9855,0.9756,0.8791,8.8662,0.8653


✅ Saved simplified summary to evaluation_results_2D_blue/metrics_simple_summary_2D_blue_20250818_221039.csv

EVALUATION COMPLETE! (2D Blue Channel Version)
Results saved to: evaluation_results_2D_blue
